# Document cost estimator

Point `INPUT_PATH` at a document, run all cells, and get:

| | |
|---|---|
| **What it is** | extracted characters and raw MB, read by the same loaders production uses |
| **What it costs** | Sonnet 5 spend in USD, from token counts measured off the live API responses |

Design: [`2026-08-12-document-cost-estimator-notebook-design.md`](../docs/superpowers/specs/2026-08-12-document-cost-estimator-notebook-design.md)

Accepts a single file (`.pdf` / `.xlsx` / `.docx` / `.txt`), a folder of them,
or a vendor `.zip`. Everything is staged into a **throwaway project under the
system temp directory** — never under `projects/`, which
`tests/test_real_corpus_coverage.py` asserts against.

`DRY_RUN = True` is the default: cells 1–6 are free and make no API calls. The
paid run is cell 7, and it tells you the projected spend before you authorise it.

In [1]:
# ── 1. CONFIG ─────────────────────────────────────────────────────────────

# The document to price: a file, a folder, or a vendor .zip.
INPUT_PATH = r"C:\Users\irahu\git_workspace\procurement-and-cost-estimation\processed-data\01-client-mr-rfq\Copy of ADN-AEC-ME-SPC-026_MR_Gas_Genset_client_MR_only.xlsx"  # "processed-data/02-vendor-bids/KERUI"

# True  -> measure and project only. No API calls, no spend.
# False -> run the real pipeline against Anthropic and measure actual tokens.
DRY_RUN = False # True

MODEL = "claude-sonnet-5"   # the only model this notebook runs or prices

# 1 MiB, so the MB column matches what a file browser reports on disk.
# Set to 1_000_000 for decimal MB.
BYTES_PER_MB = 1_048_576

VENDOR_NAME  = None   # None -> derived from INPUT_PATH
SCRATCH_ROOT = None   # None -> a fresh temp dir. Do not point this at projects/.

In [2]:
# ── 2. SETUP ──────────────────────────────────────────────────────────────
import contextlib
import logging
import os
import shutil
import sys
import tempfile
from pathlib import Path

# pypdf logs a warning per malformed cross-reference entry, and the corpus has
# thousands. They say nothing about cost; the reader that won is reported per
# document instead.
logging.getLogger("pypdf").setLevel(logging.ERROR)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "procurement").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("run this notebook from inside the repo")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

try:
    import dotenv
    dotenv.load_dotenv(REPO_ROOT / ".env")
except ImportError:
    pass

from procurement import chunking, loaders
from procurement import project as proj
from procurement.extract_requirements import REQUIREMENTS_CHUNK_CHARS
from procurement.extract_tech import TECH_CHUNK_CHARS
from procurement.pipeline import run_ingestion
from procurement.store import snapshots

# The PDF transcription fallback reads its own model from the environment, and
# it is a real Anthropic call on the same bill. Keep it on the same Sonnet.
os.environ["PDF_LLM_MODEL"] = MODEL

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 44)

print(f"repo         {REPO_ROOT}")
print(f"chunk budget tech={TECH_CHUNK_CHARS:,}  requirements={REQUIREMENTS_CHUNK_CHARS:,}")
print(f"min chars    {loaders.MIN_EXTRACTABLE_CHARS} (below this a document is recorded `failed`)")

repo         C:\Users\irahu\git_workspace\procurement-and-cost-estimation
chunk budget tech=12,000  requirements=8,000
min chars    100 (below this a document is recorded `failed`)


In [3]:
# ── 3. RATE CARD ──────────────────────────────────────────────────────────
# Sonnet 5 only. Two rows because the introductory rate expires 2026-08-31 and
# a quote written today has to survive that date.
RATE_CARD = {
    "Sonnet 5 · intro (to 2026-08-31)": {"in": 2.00, "out": 10.00},
    "Sonnet 5 · standard":              {"in": 3.00, "out": 15.00},
}
CACHE_READ_MULTIPLIER  = 0.10   # cache hits bill at 0.1x the input rate
CACHE_WRITE_MULTIPLIER = 1.25   # cache writes at 1.25x

# Pricing spec §2.2 — measured on projects/phase4c-shipped-defaults:
# 139,780 output tokens against 801,022 extracted characters.
OUTPUT_TOKENS_PER_CHAR = 139_780 / 801_022
CHARS_PER_INPUT_TOKEN  = 4


def cost_usd(rate, input_tokens, output_tokens, cache_read=0, cache_write=0):
    return (
        input_tokens * rate["in"]
        + cache_write * rate["in"] * CACHE_WRITE_MULTIPLIER
        + cache_read * rate["in"] * CACHE_READ_MULTIPLIER
        + output_tokens * rate["out"]
    ) / 1_000_000


for name, rate in RATE_CARD.items():
    print(f"{name:<34} ${rate['in']:.2f} in / ${rate['out']:.2f} out per MTok")

Sonnet 5 · intro (to 2026-08-31)   $2.00 in / $10.00 out per MTok
Sonnet 5 · standard                $3.00 in / $15.00 out per MTok


In [4]:
# ── 4. USAGE RECORDER ─────────────────────────────────────────────────────
# `classify_structure` reads `message.usage` and throws it away — that is §5.4
# of the pricing spec, and fixing it is out of scope here. So record one layer
# lower: `anthropic_client.py` constructs `anthropic.Anthropic(...)` *inside*
# the call, so the symbol resolves at call time and a swapped factory sees
# every request. `pdf_llm.transcribe_pdf` builds its client the same way, so
# its transcription spend lands in the same total.
#
# Notebook-local, installed and removed by the context manager below. Nothing
# in procurement/ or shared/ is modified.
import anthropic

_REAL_ANTHROPIC = anthropic.Anthropic


class _RecordingMessages:
    def __init__(self, inner, sink):
        self._inner = inner
        self._sink = sink

    def create(self, **kwargs):
        message = self._inner.create(**kwargs)
        usage = getattr(message, "usage", None)
        self._sink.append({
            "model": kwargs.get("model"),
            "input_tokens": getattr(usage, "input_tokens", 0) or 0,
            "output_tokens": getattr(usage, "output_tokens", 0) or 0,
            "cache_write": getattr(usage, "cache_creation_input_tokens", 0) or 0,
            "cache_read": getattr(usage, "cache_read_input_tokens", 0) or 0,
            "stop_reason": getattr(message, "stop_reason", None),
        })
        return message

    def __getattr__(self, name):
        return getattr(self._inner, name)


class _RecordingClient:
    def __init__(self, inner, sink):
        self._inner = inner
        self.messages = _RecordingMessages(inner.messages, sink)

    def __getattr__(self, name):
        return getattr(self._inner, name)


@contextlib.contextmanager
def record_usage():
    """Yield a list that fills with one entry per Anthropic request."""
    sink = []
    anthropic.Anthropic = lambda *a, **kw: _RecordingClient(_REAL_ANTHROPIC(*a, **kw), sink)
    try:
        yield sink
    finally:
        anthropic.Anthropic = _REAL_ANTHROPIC

In [5]:
# ── 5. STAGE THE INPUT ────────────────────────────────────────────────────
# One throwaway project, built under the temp directory. Everything lands on
# the vendor side: without an RFQ the compliance matrix has nothing to compare
# against, but classification, routing and extraction — the passes that cost
# money — run exactly as they do in production.
source = Path(INPUT_PATH)
if not source.is_absolute():
    source = (REPO_ROOT / source).resolve()
if not source.exists():
    raise FileNotFoundError(f"INPUT_PATH does not exist: {source}")

ROOT = Path(SCRATCH_ROOT) if SCRATCH_ROOT else Path(tempfile.mkdtemp(prefix="cost-estimate-"))
if ROOT.resolve() == (REPO_ROOT / "projects").resolve():
    raise ValueError(
        "SCRATCH_ROOT must not be projects/ — test_real_corpus_coverage.py "
        "asserts against the newest multi-vendor store there.")
ROOT.mkdir(parents=True, exist_ok=True)

project = proj.create_project(str(ROOT), f"cost estimate {source.stem}")
SLUG = project.slug
PROJECT_DIR = ROOT / SLUG

if source.suffix.lower() == ".zip":
    VENDORS = proj.unpack_vendor_zip(str(ROOT), SLUG, str(source))
else:
    vendor = proj.slugify(VENDOR_NAME or source.stem) or "vendor"
    dest = PROJECT_DIR / "vendors" / vendor
    dest.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(source, dest / source.name)
    project.vendors = proj.list_vendor_dirs(str(ROOT), SLUG)
    proj.save_project(str(ROOT), project)
    VENDORS = project.vendors

staged = [f for v in VENDORS for f in proj.vendor_files(str(ROOT), SLUG, v)]
print(f"input    {source}")
print(f"staged   {len(staged)} file(s) across {len(VENDORS)} vendor(s): {', '.join(VENDORS)}")
print(f"project  {PROJECT_DIR}")

input    C:\Users\irahu\git_workspace\procurement-and-cost-estimation\processed-data\01-client-mr-rfq\Copy of ADN-AEC-ME-SPC-026_MR_Gas_Genset_client_MR_only.xlsx
staged   1 file(s) across 1 vendor(s): copy-of-adn-aec-me-spc-026-mr-gas-genset-client-mr-only
project  C:\Users\irahu\AppData\Local\Temp\cost-estimate-n5sx61sj\cost-estimate-copy-of-adn-aec-me-spc-026-mr-gas-genset-client-mr-only


In [6]:
# ── 6. FREE PASS — characters, MB and a projected bill. No API calls. ──────
rows = []
for vendor in VENDORS:
    for path in proj.vendor_files(str(ROOT), SLUG, vendor):
        f = Path(path)
        size = f.stat().st_size
        # llm_fallback=None: the LLM transcription path cannot fire here, so
        # this pass cannot charge.
        try:
            text, reader = loaders.read_text_with_source(path, llm_fallback=None)
        except Exception as exc:
            text, reader = "", f"unreadable ({type(exc).__name__})"
        chars = len(text)
        rows.append({
            "rel": f.relative_to(PROJECT_DIR).as_posix(),
            "vendor": vendor,
            "file": f.name,
            "reader": reader,
            "bytes": size,
            "MB": round(size / BYTES_PER_MB, 3),
            "chars": chars,
            "chars/MB": round(chars / (size / BYTES_PER_MB)) if size else 0,
            "extractable": chars >= loaders.MIN_EXTRACTABLE_CHARS,
            "tech_chunks": len(chunking.chunk_on_lines(text, TECH_CHUNK_CHARS)),
            "req_chunks": len(chunking.chunk_on_lines(text, REQUIREMENTS_CHUNK_CHARS)),
        })

measured = pd.DataFrame(rows)
total_bytes = int(measured["bytes"].sum())
total_chars = int(measured["chars"].sum())

display(measured.drop(columns=["rel"]))

projected_in = total_chars / CHARS_PER_INPUT_TOKEN
projected_out = total_chars * OUTPUT_TOKENS_PER_CHAR

print(f"\n{len(measured)} file(s)   {total_bytes:,} bytes "
      f"({total_bytes / BYTES_PER_MB:.2f} MB)   {total_chars:,} chars")
print(f"not extractable: {int((~measured['extractable']).sum())} "
      f"(under {loaders.MIN_EXTRACTABLE_CHARS} chars — recorded `failed`, "
      "no extractor runs on them and they cost nothing)")
print(f"\nPROJECTED (estimate: chars÷{CHARS_PER_INPUT_TOKEN} in, "
      f"{OUTPUT_TOKENS_PER_CHAR:.4f}×chars out — corpus-measured, spec §2.2)")
print(f"  ~{projected_in:,.0f} input tokens, ~{projected_out:,.0f} output tokens")
for name, rate in RATE_CARD.items():
    print(f"  {name:<34} ${cost_usd(rate, projected_in, projected_out):,.4f}")

,vendor,file,reader,bytes,MB,chars,chars/MB,extractable,tech_chunks,req_chunks
0,copy-of-adn-aec-me-spc-026-mr-gas-genset...,Copy of ADN-AEC-ME-SPC-026_MR_Gas_Genset...,xlsx,280750,0.268,25176,94030,True,3,4



1 file(s)   280,750 bytes (0.27 MB)   25,176 chars
not extractable: 0 (under 100 chars — recorded `failed`, no extractor runs on them and they cost nothing)

PROJECTED (estimate: chars÷4 in, 0.1745×chars out — corpus-measured, spec §2.2)
  ~6,294 input tokens, ~4,393 output tokens
  Sonnet 5 · intro (to 2026-08-31)   $0.0565
  Sonnet 5 · standard                $0.0848


---
## Everything above is free. The next cell spends money.

It runs `run_ingestion` — classify → lineage → route → extract — against the
live Anthropic API on `claude-sonnet-5`, and records the token count of every
request. The projected spend is printed above.

Set `DRY_RUN = False` in cell 1 and re-run from there to authorise it.

In [7]:
# ── 7. PAID RUN — the real pipeline, with every request's usage recorded. ──
usage = []
result = None

if DRY_RUN:
    print("DRY_RUN is True — no API calls made, nothing spent.")
    print("Set DRY_RUN = False in cell 1 and re-run from there to measure for real.")
elif not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY is not set — put it in .env at the repo root.")
else:
    from procurement.pdf_llm import transcribe_pdf
    from shared.llm.anthropic_client import AnthropicClient

    client = AnthropicClient(model=MODEL)
    with record_usage() as usage:
        result = run_ingestion(str(ROOT), SLUG, client, pdf_fallback=transcribe_pdf)
    print(f"run finished: {result}")
    print(f"{len(usage)} Anthropic request(s) recorded on {MODEL}")

run finished: {'bids': [{'currency': '', 'base_price': 0.0, 'base_qty': None, 'base_unit_price': None, 'vat_included': False, 'vat_rate': 0.0, 'freight_amount': 0.0, 'freight_included': False, 'discount_pct': 0.0, 'optional_items': [{'description': 'Selective Catalytic Reduction system (SCR)', 'qty': None, 'unit_price': None, 'total': None, 'category': 'emission control', 'included_in_base': False}, {'description': 'Pre-commissioning/Commissioning/Start-up spares', 'qty': None, 'unit_price': None, 'total': None, 'category': 'spares', 'included_in_base': False}, {'description': 'Two Years Operation Spares', 'qty': None, 'unit_price': None, 'total': None, 'category': 'spares', 'included_in_base': False}, {'description': 'Special Tools and Tackles for Erection, Pre-commissioning, Commissioning & Operation', 'qty': None, 'unit_price': None, 'total': None, 'category': 'tools', 'included_in_base': False}], 'delivery_time': None, 'delivery_terms': None, 'payment_terms': None, 'engine_make': N

In [8]:
# ── 8. COST REPORT ────────────────────────────────────────────────────────
documents = snapshots.load_documents(str(ROOT), SLUG) if not DRY_RUN else []
by_path = {d.path: d for d in documents}

report = measured.copy()
if documents:
    report["class"] = report["rel"].map(lambda p: getattr(by_path.get(p), "doc_class", "—"))
    report["source"] = report["rel"].map(lambda p: getattr(by_path.get(p), "text_source", None) or "—")
    report["status"] = report["rel"].map(lambda p: getattr(by_path.get(p), "extraction_status", "not ingested"))
    report["note"] = report["rel"].map(lambda p: (getattr(by_path.get(p), "notes", None) or "")[:60])
    columns = ["vendor", "file", "class", "reader", "source", "MB", "chars", "status", "note"]
else:
    # Nothing has been classified or extracted, so those columns would be a
    # wall of em-dashes. Show what the free pass actually knows instead.
    columns = ["vendor", "file", "reader", "MB", "chars", "chars/MB",
               "extractable", "tech_chunks"]

display(report[columns])

# ── tokens ───────────────────────────────────────────────────────────────
if usage:
    calls = pd.DataFrame(usage)
    tok_in = int(calls["input_tokens"].sum())
    tok_out = int(calls["output_tokens"].sum())
    cache_read = int(calls["cache_read"].sum())
    cache_write = int(calls["cache_write"].sum())
    token_basis = f"MEASURED across {len(calls)} request(s)"
    truncated = calls[calls["stop_reason"] == "max_tokens"]
    if len(truncated):
        print(f"!! {len(truncated)} response(s) hit the max_tokens ceiling — "
              "the extraction they belong to failed. See DocumentRecord.notes.")
else:
    tok_in, tok_out, cache_read, cache_write = projected_in, projected_out, 0, 0
    token_basis = "PROJECTED (chars÷4 in, corpus ratio out)"

print(f"\nTOKENS — {token_basis}")
print(f"  input {tok_in:,.0f}   output {tok_out:,.0f}   "
      f"cache read {cache_read:,}   cache write {cache_write:,}")
if usage:
    print(f"  vs projection: input {projected_in:,.0f} ({tok_in / projected_in:.2f}×), "
          f"output {projected_out:,.0f} ({tok_out / projected_out:.2f}×)"
          if projected_in and projected_out else "")

# ── cost ─────────────────────────────────────────────────────────────────
print("\nCOST (USD)")
display(pd.DataFrame([
    {"rate card": name,
     "USD": round(cost_usd(rate, tok_in, tok_out, cache_read, cache_write), 4)}
    for name, rate in RATE_CARD.items()
]))

,vendor,file,class,reader,source,MB,chars,status,note
0,copy-of-adn-aec-me-spc-026-mr-gas-genset...,Copy of ADN-AEC-ME-SPC-026_MR_Gas_Genset...,spec,xlsx,xlsx:25176chars,0.268,25176,ok,This document is a Material Requisition ...



TOKENS — MEASURED across 4 request(s)
  input 28,536   output 4,946   cache read 0   cache write 0
  vs projection: input 6,294 (4.53×), output 4,393 (1.13×)

COST (USD)


,rate card,USD
0,Sonnet 5 · intro (to 2026-08-31),0.1065
1,Sonnet 5 · standard,0.1598


In [9]:
# ── 9. CLEAN UP ───────────────────────────────────────────────────────────
# The scratch project holds a copy of the input and the extracted store.
# Re-staging from scratch is also how you get a clean second measurement:
# run_ingestion caches on content_sha256, so a second run of the same bytes
# measures almost no tokens.
if SCRATCH_ROOT is None:
    shutil.rmtree(ROOT, ignore_errors=True)
    print(f"removed {ROOT}")
else:
    print(f"kept {ROOT} (SCRATCH_ROOT was set explicitly)")

removed C:\Users\irahu\AppData\Local\Temp\cost-estimate-n5sx61sj
